In [ ]:
# https://github.com/h2oai/sql-sidekick/releases
# !python -m pip uninstall sql_sidekick-0.2.4-py3-none-any.whl -y

In [ ]:
import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

In [ ]:
from sidekick.prompter import db_setup, ask
from sidekick.schema_generator import generate_schema
from sidekick.utils import setup_dir, list_models

In [ ]:
base_path = "./"
cache_path = f"{base_path}/var/lib/tmp"
setup_dir(base_path)

In [ ]:
# import pandas as pd
# f = pd.read_csv("../demo/demo_data.csv")

In [ ]:
# Env variables
import os

os.environ['OPENAI_API_KEY'] = ""
os.environ['H2OGPT_URL'] = ""
os.environ['H2OGPT_API_TOKEN'] = ""
# To get access to h2ogpte endpoint, reach out to cloud-feedback@h2o.ai
os.environ['H2OGPTE_URL'] = ""  # e.g. https://<>.h2ogpte.h2o.ai
os.environ['H2OGPTE_API_TOKEN'] = ""

os.environ['H2OGPT_BASE_URL'] = ""

In [ ]:
HOST_NAME = "localhost"
USER_NAME = "sqlite"
PASSWORD = "abc"
DB_NAME = "querydb"
PORT = "5432"


# Given .csv file, auto-generate schema
# Download dataset --> https://www.kaggle.com/datasets/uom190346a/sleep-health-and-lifestyle-dataset
data_path = "examples/demo/demo_data.csv"
table_name = "sleep_health_eda"

r, table_info_path = generate_schema(data_path=data_path, output_path=f"{cache_path}/{table_name}_table_info.jsonl")

llm_response, err = db_setup(
                db_name=DB_NAME,
                hostname=HOST_NAME,
                user_name=USER_NAME,
                password=PASSWORD,
                port=PORT,
                table_info_path=table_info_path,
                table_samples_path=data_path,
                table_name=table_name,
                local_base_path=base_path
            )

In [ ]:
# List supported models
list_models()

In [ ]:
# Use `list_models()` to check other supported models

def query(
    question: str,
    table_name,
    table_info_path: str,
    sample_qna_path: str,
    regenerate=False,
    regenerate_with_options=False,
):
    """Asks question and returns SQL."""
    base_path = "./"
    # self_correction is enabled by default, set to False if not needed.
    res = ask(
        question=question,
        table_info_path=table_info_path,
        sample_queries_path=sample_qna_path,
        table_name=table_name,
        is_command=False,
        model_name="gpt-4o",  #Other default model option: h2ogpt-sql-sqlcoder-34b-alpha
        is_regenerate=regenerate,
        is_regen_with_options=regenerate_with_options,
        execute_query=False,
        local_base_path=base_path,
    )
    return res

In [ ]:
res = query("What is the average sleep duration for each gender?", table_name="sleep_health_eda",
            table_info_path=table_info_path, sample_qna_path=None)

In [ ]:
print(f"Question = {res[0][0]}")
print("----")
print(f"Generated SQL = {res[0][1]}")

In [ ]:
# On using re-generation flag we toggle the temperature values between 0 and 1 alternating between low
# (focus/conservative generation and high values (random/creative generation)
res = query("What are the most common occupations among individuals in the dataset?", table_name="sleep_health_eda",
            table_info_path=table_info_path, sample_qna_path=None, regenerate=True)

In [ ]:
print(f"Question = {res[0][0]}")
print("----")
print(f"Generated SQL = {res[0][1]}")

In [ ]:
# Alternate options
res = query("What is the average sleep duration for each gender?", table_name="sleep_health_eda",
            table_info_path=table_info_path, sample_qna_path=None, regenerate_with_options=True)

In [ ]:
print(f"Question = {res[0][0]}")
print("----Options----")
for _r in res[1]:
    print(_r)